# Purpose
- create PSScene orders for key sites
- download PSScene orders for key sites

# Imports


In [1]:
import errno
import json
import os
import sys
import time
from pathlib import Path
import csv
from tqdm import  tqdm
import pandas as pd
import requests
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

True

In [2]:
# TODO check if order already created, example if site1_chunk0_2025_2026 was already created no need to create new order
# since not really repeating I think find to skip for now

# Options

In [3]:
SITE_TO_PROCESS = "Willard_Juniper_Savannah"

ORDERS_MIN_YEAR = 2023 # inclusive
ORDERS_MAX_YEAR = 2026 # exclusive

CREATE_ORDERS = False
DOWNLOAD_ORDERS = True

In [4]:
print(f'running for [{ORDERS_MIN_YEAR} - {ORDERS_MAX_YEAR}) {SITE_TO_PROCESS} with {CREATE_ORDERS=} and {DOWNLOAD_ORDERS=}')

running for [2023 - 2026) Willard_Juniper_Savannah with CREATE_ORDERS=False and DOWNLOAD_ORDERS=True


# Setup

In [5]:
# Planet urls

base_url   = "https://api.planet.com/data/v1"
orders_url = 'https://api.planet.com/compute/ops/orders/v2'
stats_url  = "{}/stats".format(base_url)
quick_url  = "{}/quick-search".format(base_url)

In [6]:
BASE_PATH = Path('/projectnb/planet/PLSP')
NEW_BASE_PATH = Path('/projectnb/modislc/users/fache/data/planet/')
OUTPUT_DIR = NEW_BASE_PATH / 'raw'

# raw imagery in NEW_BASE_PATH / raw
# per site data in NEW_BASE_PATH / raw / site
# includes quick search results, order metadata, order results, data dir (per chunk)

# Site Metadata

In [7]:
def p(data):
    print(json.dumps(data, indent=2))

def create_geojson_file(row):
    return BASE_PATH / 'geojson' / f'{row["site"]}.geojson'

def create_raw_path(row):
    return BASE_PATH / 'raw' / row["site"]

def create_test_raw_path(row):
    return NEW_BASE_PATH / 'raw' / row["site"]

In [8]:
metadata_df = pd.DataFrame()
metadata_df['site'] = ['Walnut_Gulch_Kendall_Grasslands', 'Willard_Juniper_Savannah', 'Mountainair_Pinyon-Juniper_Woodland', 'Santa_Rita_Grassland', 'Santa_Rita_Mesquite', 'Sevilleta_shrubland', 'Walnut_Gulch_Lucky_Hills_Shrub', 'ARM_Southern_Great_Plains_site-_Lamont']
metadata_df['geojson_file'] = metadata_df.apply(create_geojson_file, axis=1)
metadata_df['raw_path'] = metadata_df.apply(create_raw_path, axis=1)
metadata_df['test_raw_path'] = metadata_df.apply(create_test_raw_path, axis=1)
metadata_df.head(10)

,site,geojson_file,raw_path,test_raw_path
0,Walnut_Gulch_Kendall_Grasslands,/projectnb/planet/PLSP/geojson/Walnut_Gulch_Ke...,/projectnb/planet/PLSP/raw/Walnut_Gulch_Kendal...,/projectnb/modislc/users/fache/data/planet/raw...
1,Willard_Juniper_Savannah,/projectnb/planet/PLSP/geojson/Willard_Juniper...,/projectnb/planet/PLSP/raw/Willard_Juniper_Sav...,/projectnb/modislc/users/fache/data/planet/raw...
2,Mountainair_Pinyon-Juniper_Woodland,/projectnb/planet/PLSP/geojson/Mountainair_Pin...,/projectnb/planet/PLSP/raw/Mountainair_Pinyon-...,/projectnb/modislc/users/fache/data/planet/raw...
3,Santa_Rita_Grassland,/projectnb/planet/PLSP/geojson/Santa_Rita_Gras...,/projectnb/planet/PLSP/raw/Santa_Rita_Grassland,/projectnb/modislc/users/fache/data/planet/raw...
4,Santa_Rita_Mesquite,/projectnb/planet/PLSP/geojson/Santa_Rita_Mesq...,/projectnb/planet/PLSP/raw/Santa_Rita_Mesquite,/projectnb/modislc/users/fache/data/planet/raw...
5,Sevilleta_shrubland,/projectnb/planet/PLSP/geojson/Sevilleta_shrub...,/projectnb/planet/PLSP/raw/Sevilleta_shrubland,/projectnb/modislc/users/fache/data/planet/raw...
6,Walnut_Gulch_Lucky_Hills_Shrub,/projectnb/planet/PLSP/geojson/Walnut_Gulch_Lu...,/projectnb/planet/PLSP/raw/Walnut_Gulch_Lucky_...,/projectnb/modislc/users/fache/data/planet/raw...
7,ARM_Southern_Great_Plains_site-_Lamont,/projectnb/planet/PLSP/geojson/ARM_Southern_Gr...,/projectnb/planet/PLSP/raw/ARM_Southern_Great_...,/projectnb/modislc/users/fache/data/planet/raw...


## Connect


In [9]:
key = os.environ.get("PLANET_API_KEY")
print("key exists:", key is not None)
print("key length:", len(key) if key else 0)
# print("key repr:", repr(key))

PLANET_API_KEY = os.getenv('PLANET_API_KEY')

# Setup the session
session = requests.Session()
# Authenticate
session.auth = (PLANET_API_KEY, "")

print("GET:", session.get("https://api.planet.com/data/v1").status_code)

# Make a GET request to the Planet Data API
res = session.get(base_url)
# Response status code
if res.status_code != 200:
    print("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
    sys.exit("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
else:
    print("Base server is alive.")

p(res.json())

key exists: True
key length: 36
GET: 200
Base server is alive.
{
  "_links": {
    "_self": "https://api.planet.com/data/v1/",
    "asset-types": "https://api.planet.com/data/v1/asset-types/",
    "item-types": "https://api.planet.com/data/v1/item-types/",
    "spec": "https://api.planet.com/data/v1/spec"
  }
}


# Create Orders - Helpers


In [10]:
def setup_filter(coords, minyear, maxyear):
    
    geometry_filter = {
        "type": "GeometryFilter",
        "field_name": "geometry",
        "config": {
          "type": "Polygon",
          "coordinates": coords
        }
    }
    
    date_filter = {
        "type": "DateRangeFilter",
        "field_name": "acquired", # date on which the "image was taken"
        "config":    {
            "gte": "{}-01-01T00:00:00.000Z".format(minyear),
            "lt":"{}-01-01T00:00:00Z".format(maxyear)
        }
    }
    
    ground_control =  {
        "type": "StringInFilter",
        "config": ["true"],
        "field_name": "ground_control" # NOTE
    }
    
    quality_category = {
        "type": "StringInFilter",
        "config": ["standard"],
        "field_name": "quality_category" # NOTE
    }
    
    cloud_cover =  {
        "type": "RangeFilter",
        "field_name": "cloud_cover",
        "config": {
            "gte": 0,
            "lte": 0.5
        } # NOTE
    }
    
    asset = {
        "type": "AssetFilter",
        "config": [
            "ortho_analytic_4b_sr", # NOTE before analytic_sr 
            "ortho_analytic_4b", # NOTE before analytic
            "ortho_udm2" # NOTE udm2 no longer exists, for instance, is only available globally through July 2018."
        ]
    }

    permission = {
        "type":"PermissionFilter",
        "config": [
            "assets:download" # NOTE
        ]
    }

    and_filter = {
        "type": "AndFilter",
        "config": [ 
            cloud_cover,
            quality_category,
            ground_control,
            date_filter,
            geometry_filter,
            permission,
            asset
        ]
    }
    
    return and_filter

def read_geometry(path):
    
    if not os.path.exists(path):
        sys.exit("GeoJSON path doesn't exist: {}".format(path))#
        
    with open(path, "r") as f:
        geo = json.load(f)
        
    return geo

def place_order(request, auth, order_name):
    headers = {'content-type': 'application/json'}
    
    response = requests.post(orders_url, data=json.dumps(request), auth=auth, headers=headers)
    print(f'{response=}')
    print(f'{response.reason=}')
    
    if response.status_code != 202:
        print("Order failed for {}".format(order_name))
        print(f'{response.text=}')
        return -1
    
    order_id = response.json()['id']

    # global GLOBAL_ORDER_ID
    # GLOBAL_ORDER_ID += 1

    # order_id = str(GLOBAL_ORDER_ID)

    print(f'{order_id=}')
    order_url = orders_url + '/' + order_id
    return order_url

# Create Orders


In [11]:
if CREATE_ORDERS:
    row = metadata_df[metadata_df['site'] == SITE_TO_PROCESS].iloc[0]

    start_time = time.time()
    print(f'{"="*10} {row["site"]} {"="*10}')
    print(f'{ORDERS_MIN_YEAR=} {ORDERS_MAX_YEAR=}')

    output_site_dir = os.path.join(OUTPUT_DIR, row['site'])
    if not os.path.exists(output_site_dir):
        try:
            os.makedirs(output_site_dir)
        except OSError as exc: # Guard against race condition
            if exc.errno != errno.EEXIST:
                raise
    
    print(f'{output_site_dir=}')

    geo = read_geometry(row['geojson_file'])
    
    for feature_num, x in enumerate(geo['features']): # get all geometry features, for each one # NOTE not really necessary since just one feature for each site
        print(f'\n{feature_num=}')

        feature_name = x['properties']['f']
        feature_coords = x['geometry']['coordinates']
        print(f'{feature_name=}')
        
        filter = setup_filter(feature_coords, ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)
        
        # print("Filter config for search:")
        # p(filter)

        # ---------- get some quick stats
        
        print("---- feature count at year interval ----")
        request = {
            "interval" : "year",
            "filter" : filter,
            "item_types" : ["PSScene"] # NOTE replaces PSScene4Band https://community.planet.com/product-updates/event-psscene-migration-workshop-161
        }

        # Send the POST request to the API stats endpoint
        res = session.post(stats_url, json=request)

        # print(res.status_code)
        # print(res.text)
        # print(res.request.headers)
        # print(res.request.body)
        
        if res.status_code != 200:
            sys.exit("Stats search failed with code {}".format(res.status_code))

        for bucket in res.json()['buckets']: # 1 bucket per interval, ex 1 aggregate bucket per year
            print("start_time: {} count: {}".format(bucket["start_time"], bucket["count"]))
        
        # ---------- perform real asset search

        print("---- quick search ----")
        request = {
            "filter" : filter,
            "item_types" : ["PSScene"]
        }

        # Send the POST request to the API quick search endpoint
        res = session.post(quick_url, json=request)
        if res.status_code != 200:
            sys.exit("Quick search failed with code {}".format(res.status_code))
        quick_search_results_json = res.json()
        
        filename = "{}_quick_search_result_{}_{}.json".format(feature_name.replace(" ", "_"), ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)
        with open(os.path.join(output_site_dir, filename), 'w') as outfile:
            json.dump(quick_search_results_json, outfile)
            print('quick-search output file created: {}'.format(filename))
        
        # ---------- get all assets that need to be downloaded

        print('---- assembling feature ids to download ----')
        features = quick_search_results_json['features']

        if len(features) == 0:
            sys.exit("0 IDs returned in quick search.")

        id_list = []
        num_next_urls = 0
        while len(quick_search_results_json["features"]) > 0: # loop through _next url pagination
            print('iteration: {}'.format(num_next_urls))
            
            for x in quick_search_results_json["features"]: # go through all features and collect all scene ids
                id_list.append(x["id"])
            
            # Assign the "_links" -> "_next" property (link to next page of results) to a variable 
            next_url = quick_search_results_json["_links"]["_next"]
            if next_url is None:
                break
            num_next_urls += 1
            
            # from the next url, if there are results, update quick_search_results_json and append to output_site_dir
            time.sleep(5)
            res = session.get(next_url)
            
            if res.status_code != 200:
                sys.exit("Next page retrieval failed with code {}".format(res.status_code))
                
            quick_search_results_json = res.json()
            with open(os.path.join(output_site_dir, filename), 'a') as outfile:
                json.dump(quick_search_results_json, outfile)
            
            # output_site_dir is now on the next page, keep looping for more features
        
        # ---- end quick search loop

        print("total feature ids: {}".format(len(id_list)))
        print(f'{num_next_urls=}')

        print('---- chunking ids ----')
        chunks = [id_list[x:x+400] for x in range(0, len(id_list), 400)]
        
        print(f'{len(chunks)=}')
        
        if len(chunks) >= 80: # NOTE
            sys.exit("{} Chunks which is greater than 80.  This will exceed order capacity".format(len(chunks)))


        print("---- Checking connection with order server... ----")
        auth = HTTPBasicAuth(PLANET_API_KEY, '')
        response = requests.get(orders_url, auth=auth)
        
        if response.status_code != 200:
            sys.exit("Failed to connect to order server with code {}".format(response.status_code))
        else:
            print("connected!")
            
        orders_list = response.json()["orders"] # returns all previous orders created through my API key

        print('---- placing orders ----')
        # iterate through chunks (each is list of features (partial scenes))
        # create order per chunk
        # buffer coordinates
        # create order dir
        # place order

        orders_url_list = []
        bad_order_count = 0

        for chunk_num, chunk in enumerate(chunks):
            print(f'processing chunk {chunk_num}')

            order_name = "{}_chunk_{}_{}_{}".format(feature_name.replace(" ", "_"), chunk_num, ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)

            # expand the coordinates by 0.015 degree rectangle
            feature_coords_buffer = feature_coords
            feature_coords_buffer[0][0][0] = feature_coords_buffer[0][0][0] - 0.0015 # top left (lat, lon)
            feature_coords_buffer[0][0][1] = feature_coords_buffer[0][0][1] + 0.0015    
            feature_coords_buffer[0][1][0] = feature_coords_buffer[0][1][0] + 0.0015 # top right
            feature_coords_buffer[0][1][1] = feature_coords_buffer[0][1][1] + 0.0015
            feature_coords_buffer[0][2][0] = feature_coords_buffer[0][2][0] + 0.0015 # bottom right
            feature_coords_buffer[0][2][1] = feature_coords_buffer[0][2][1] - 0.0015
            feature_coords_buffer[0][3][0] = feature_coords_buffer[0][3][0] - 0.0015 # bottom left
            feature_coords_buffer[0][3][1] = feature_coords_buffer[0][3][1] - 0.0015
            feature_coords_buffer[0][4][0] = feature_coords_buffer[0][4][0] - 0.0015 # top left
            feature_coords_buffer[0][4][1] = feature_coords_buffer[0][4][1] + 0.0015

            request = {
                "name": order_name,
                "order_type": "partial",
                "products": [
                    {
                        "item_ids": chunk, # item ids belonging to this chunk, ids are the features found from the quick search
                        "item_type": "PSScene", # NOTE changed from PSScene4Band
                        "product_bundle": "analytic_sr_udm2"
                        # analytic_sr_udm2 - now includes standard UDM2 mask for PSScene
                        # https://docs.planet.com/data/imagery/udm/
                        # correct band version is downloaded for provided item_type
                        # https://docs.planet.com/data/imagery/planetscope/#surface-reflectance
                        
                        # NOTE changed from analytic_sr_udm2,analytic_sr
                        # analytic_udm2 contains [ortho_analytic_4b, ortho_analytic_4b_xml, ortho_udm2]
                        # analytic_sr_udm2 contains [ortho_analytic_4b_sr, ortho_analytic_4b_xml, ortho_udm2]
                    }
                ],
                "tools": [
                    {
                        "clip": {     
                            "aoi": {
                                "type": "Polygon",
                                "coordinates": feature_coords_buffer
                            }
                        }
                    }
                ]
            }

            filename = "order_{}.json".format(order_name)
            with open(os.path.join(output_site_dir, filename), 'w') as outfile:
                json.dump(request, outfile)
                print('orders output file created: {}'.format(filename))
            
            print('PLACING ORDER')
            
            time.sleep(5)
            
            order_result = place_order(request, auth, order_name)
            
            if order_result == -1:
                bad_order_count = bad_order_count +1
                continue
            else:
                print("order url: {}".format(order_result))
                orders_url_list.append(order_result)
        
        # ---- end chunking loop

        print("\n{} out of {} orders placed successfully.".format(len(chunks) - bad_order_count, len(chunks)))      
        if len(orders_url_list) == 0:
            sys.exit("No orders placed successfully.")

        # save order urls to csv
        filename = "{}_orders_url_result_{}_{}.csv".format(feature_name.replace(" ", "_"), ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)
        with open(os.path.join(output_site_dir, filename), 'w', newline='') as outfile:
            writer = csv.writer(outfile)
            writer.writerows([[url] for url in orders_url_list])
            print('order urls file created: {}'.format(filename))

    print("---- %.2f seconds ----" % (time.time() - start_time))
else:
    print("NO ORDERS PLACED")

NO ORDERS PLACED


# Download Orders - Helpers

In [12]:
# to simplify all of the waiting, verify order is success in ui at
# https://insights.planet.com/data/orders/
def get_download_request(order_url, auth):
    response = requests.get(order_url, auth=auth)

    print(f'{response=}')
    print(f'{response.status_code=}')

    state = response.json()['state']
    print(f'{state=}')

    return response



def download_results(results, OUTPUT_DIR, order_name, overwrite=False):
    results_urls = [r['location'] for r in results] # individual feature urls (slices)
    results_names = [r['name'] for r in results] # file names, ex 91ed005e-1d5a-4ecd-9be9-3fc9c1f53afb/PSScene/20250110_181620_58_24fa_3B_AnalyticMS_SR_clip.tif
    print('{} items to download'.format(len(results_urls)))

    # NOTE item count per order is chunk size (400 or remainder) x 4 (3B_AnalyticMS_metadata_clip.xml, 3B_AnalyticMS_SR_clip.tif, metadata.json, 3B_udm2_clip.tif) + 1 (manifest.json)
    
    failed_count = 0
    skipped_count = 0
    success_count = 0
    
    filename = "failed_downloads_{}.txt".format(order_name)
    
    with open(os.path.join(OUTPUT_DIR, filename), "w") as file1:
    
        # go through each order urls results urls
        for url, name in tqdm(zip(results_urls, results_names)):
            path = Path(os.path.join(OUTPUT_DIR, 'data', name))

            if overwrite or not path.exists():
                print('downloading {} to {}'.format(name, path))
                #logging.info('downloading {} to {}'.format(name,path))
                r = requests.get(url, allow_redirects=True)
                if(r.status_code == 200):
                    path.parent.mkdir(parents=True, exist_ok=True)
                    open(path, 'wb').write(r.content)
                    success_count += 1
                else:
                    #logging.error('Status code {}, {} not downloaded.')
                    print('Status code {}, {} not downloaded.'.format(r.status_code, name))
                    failed_count += 1
                    file1.write("{}, {}, {}, {} \n".format(time.strftime("%Y%m%d-%H%M%S"), r.status_code, name, url))
            else:
                print('{} already exists, skipping {}'.format(path, name))
                skipped_count += 1
                #logging.info('{} already exists, skipping {}'.format(path, name))
    
    print('STATS')
    print("\n{} - Success: {} Skipped: {} Failed: {}".format(order_name, success_count, skipped_count, failed_count))
    
    return failed_count, skipped_count, success_count


# Download Orders

In [ ]:
if DOWNLOAD_ORDERS:

    print("---- checking connection with order server... ----")
    auth = HTTPBasicAuth(PLANET_API_KEY, '')
    response = requests.get(orders_url, auth=auth)
    
    if response.status_code != 200:
        sys.exit("Failed to connect to order server with code {}".format(response.status_code))
    else:
        print("connected!")

    row = metadata_df[metadata_df['site'] == SITE_TO_PROCESS].iloc[0]


    start_time = time.time()
    print(f'\n\n\n\n{"="*40}\n{"="*40}')
    print(f'{"="*10} {row["site"]} {"="*10}')
    print(f'{ORDERS_MIN_YEAR=} {ORDERS_MAX_YEAR=}')

    output_site_dir = os.path.join(OUTPUT_DIR, row['site'])

    feature_name = row["site"]

    total_files_to_download = 0
    total_failed = 0
    total_skipped = 0
    total_success = 0
    order_list_failed = []
    order_list_failed_download = []

    # read from saved orders urls
    filename = "{}_orders_url_result_{}_{}.csv".format(feature_name.replace(" ", "_"), ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)
    df = pd.read_csv(os.path.join(output_site_dir, filename), header=None)
    orders_url_list = df.values.flatten().tolist()
    print(f'{len(orders_url_list)} orders to monitor')

    for chunk_num, order_url in enumerate(orders_url_list):
        try:
            order_name = "{}_chunk_{}_{}_{}".format(feature_name.replace(" ", "_"), chunk_num, ORDERS_MIN_YEAR, ORDERS_MAX_YEAR)
            print('MONITORING')
            print("Monitoring order number {} (url: {})".format(chunk_num, order_url))
            
            r = get_download_request(order_url, auth=auth)
            response = r.json()
            state = response['state']
            
            if state != "success" :
                print("Order not success, status is {}".format(state))
                order_list_failed.append(chunk_num)
                continue

            results = response['_links']['results']
            
            print("Total files to download: {} ".format(len(results)))
            
            total_files_to_download += len(results)
            
            filename = "order_result_{}.json".format(order_name)
            with open(os.path.join(output_site_dir, filename), 'w') as outfile:
                json.dump(response, outfile)
                print('order result file created: {}'.format(filename))

            print(f'==== downloading files for chunk {chunk_num} ====')
            failed_count, skipped_count, success_count = download_results(results, output_site_dir, order_name)
            
            if failed_count != 0:
                order_list_failed_download.append(chunk_num)
            
            total_failed += failed_count
            total_skipped += skipped_count
            total_success += success_count

        except Exception as e:
            print(f'!!!! error with {chunk_num} !!!!')
            print(e)
            pass

    print("\nSUMMARY")
    print("Total Files to be downloaded: {}".format(total_files_to_download))
    print("Total Files Failed to Download: {}".format(total_failed))
    print("Total Files Skipped: {}".format(total_skipped))
    print("Total Files Success: {}".format(total_success))
    print("Order Chunks Failed: {}".format(order_list_failed))
    print("Download Chunks Failed: {}".format(order_list_failed_download))
    print("--- %.2f seconds ---" % (time.time() - start_time))
else:
    print("NO DOWNLOADS")

---- checking connection with order server... ----
connected!




========== Willard_Juniper_Savannah ==========
ORDERS_MIN_YEAR=2023 ORDERS_MAX_YEAR=2026
7 orders to monitor
MONITORING
Monitoring order number 0 (url: https://api.planet.com/compute/ops/orders/v2/fa0581e2-53ad-4c8b-adcb-684a9d6b0567)
response=<Response [200]>
response.status_code=200
state='success'
Total files to download: 1601 
order result file created: order_result_Willard_Juniper_Savannah_chunk_0_2023_2026.json
==== downloading files for chunk 0 ====
1601 items to download


0it [00:00, ?it/s]

downloading fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251214_175035_98_255d_metadata.json to /projectnb/modislc/users/fache/data/planet/raw/Willard_Juniper_Savannah/data/fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251214_175035_98_255d_metadata.json


1it [00:00,  2.33it/s]

downloading fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251214_175035_98_255d_3B_udm2_clip.tif to /projectnb/modislc/users/fache/data/planet/raw/Willard_Juniper_Savannah/data/fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251214_175035_98_255d_3B_udm2_clip.tif


...

downloading fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_udm2_clip.tif to /projectnb/modislc/users/fache/data/planet/raw/Willard_Juniper_Savannah/data/fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_udm2_clip.tif


66it [00:31,  1.67it/s]

downloading fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_AnalyticMS_metadata_clip.xml to /projectnb/modislc/users/fache/data/planet/raw/Willard_Juniper_Savannah/data/fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_AnalyticMS_metadata_clip.xml


67it [00:32,  1.94it/s]

downloading fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_AnalyticMS_SR_clip.tif to /projectnb/modislc/users/fache/data/planet/raw/Willard_Juniper_Savannah/data/fa0581e2-53ad-4c8b-adcb-684a9d6b0567/PSScene/20251128_181733_43_2540_3B_AnalyticMS_SR_clip.tif
